In [ ]:
import os 
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [3]:
from langchain.chat_models import init_chat_model
from langchain_google_genai import  ChatGoogleGenerativeAI

In [4]:
model_groq = init_chat_model("groq:llama-3.1-8b-instant")
model_google = ChatGoogleGenerativeAI(model="gemini-3-flash-preview")

In [6]:
model_google.invoke("good norning").text

'Good morning! I hope your day is off to a great start. How can I help you today?'

In [7]:
model_groq.invoke("good norning").content

'Good morning. Is there something I can help you with, or would you like to chat?'

### Tools

In [25]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get weather at a location"""
    return f"weather is sunny at {location}"

model_with_tool = model_google.bind_tools([get_weather])

In [15]:
res = model_with_tool.invoke("what is weather at Banglore")
res

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'jch8nnjzt', 'function': {'arguments': '{"location":"Banglore"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 216, 'total_tokens': 231, 'completion_time': 0.028173804, 'completion_tokens_details': None, 'prompt_time': 0.017857545, 'prompt_tokens_details': None, 'queue_time': 0.050701628, 'total_time': 0.046031349}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a008f1-d5bd-7223-9615-6fbc587e1011-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Banglore'}, 'id': 'jch8nnjzt', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 216, 'output_tokens': 15, 'total_tokens': 231})

In [18]:
res.tool_calls

[{'name': 'get_weather',
  'args': {'location': 'Banglore'},
  'id': 'b3qg7d1w0',
  'type': 'tool_call'}]

### Tool Execution Loop

In [28]:
# step 1: model generate tool call

message = [{"role":"user","content":"what is weather at Banglore"}]
ai_msg = model_with_tool.invoke(message)
message.append(ai_msg)

# step 2: Execute Tools And result
for tool_call in ai_msg.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    message.append(tool_result)

#pass the result for final responce
final_result = model_with_tool.invoke(message)
#print(final_result)


final_result.text

'OK. The weather in Bangalore is currently sunny.'

In [29]:
message

[{'role': 'user', 'content': 'what is weather at Banglore'},
 AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "Bangalore"}'}, '__gemini_function_call_thought_signatures__': {'call_2173277': 'Eu4BCusBARFNMg9lTRjjd3ATX2U/OLi+2D/EJuTxag4I+o3PIu0D33YAfkM4Qrdv4u4V+azE8JqlBKQzHPsFhGRvwlruK0xp3XXFhf3fRifLpz5ie20aLf3L+bwrRRY5cpHJub/ih4nbKyBoOZZS8+MvzIp20zm+TFElp0C7f90O9teS9gDmF9vBZ7I511cY1gT7D3WGdLQRd7pFElbBAFBG+km5sxHEg+PeCnL4bb8+jddR5fnGpdbr7ibGylGetghGf6rzUombkmPnIyWF1D+21+oaxZpJczMRH2gXhdSBIDXMOTc5Wg/nhMIssPoatA=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3-flash-preview', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0091c-e1b3-7622-9468-21752a907d2e-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Bangalore'}, 'id': 'call_2173277', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 49, 'output_tokens': 45, 'total_tokens': 94, 'input_to